# Buy or Bye — EDA ve Modelleme

UCI Online Shoppers verisi üzerinde yeniden üretilebilir analiz. Model tuning yalnız train setinde 5-fold CV PR-AUC ile yapılmış; calibration OOF tahminlerle kurulmuş ve test seti yalnız final raporlamada kullanılmıştır.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data import load_data, get_feature_groups
df = load_data(ROOT / 'data/raw/online_shoppers_intention.csv')
groups = get_feature_groups(df)
print(f'Satır: {len(df):,} | Sütun: {df.shape[1]} | Eksik: {df.isna().sum().sum()}')
df.head()

Satır: 12,205 | Sütun: 18 | Eksik: 0


In [ ]:
df['Revenue'].value_counts().rename(index={False: 'Bye', True: 'Buy'}).to_frame('Oturum')

## EDA görselleri

![Hedef dağılımı](../outputs/target_distribution.png)

![Korelasyon](../outputs/correlation_heatmap.png)

In [ ]:
df[groups['numeric']].describe().T.round(3)

## Eğitim tasarımı

Aşağıdaki komut 70/15/15 stratified split, train üzerinde 5-fold tuning, OOF sigmoid calibration, maliyet-duyarlı threshold, ablation, multi-seed ve temporal proxy analizlerini çalıştırır.

```bash
python -m src.tune
```

In [2]:
selection = json.loads((ROOT / 'reports/feature_engineering_report.json').read_text())
rows = selection.get('tuned_engineered_results', selection.get('cv_results'))
pd.DataFrame(rows)[['model','cv_pr_auc_mean','cv_pr_auc_std','cv_train_pr_auc_mean']].round(4)

model,cv_pr_auc_mean,cv_pr_auc_std,cv_train_pr_auc_mean
Engineered Random Forest,0.7529,0.0077,0.8837
Engineered LightGBM,0.7562,0.0080,0.8614


In [3]:
final_report = json.loads((ROOT / 'reports/ensemble_report.json').read_text())
test = final_report['test_metrics']
selected = 'RF + Engineered LightGBM Ensemble'
pd.Series(test)[['roc_auc','pr_auc','precision_optimized','recall_optimized','f1_optimized','threshold_optimized']].to_frame(f'{selected} test').round(4)

,RF + Engineered LightGBM Ensemble test
roc_auc,0.9320
pr_auc,0.7448
precision_optimized,0.5870
recall_optimized,0.7902
f1_optimized,0.6736
threshold_optimized,0.2500


## Açıklanabilirlik

![SHAP summary](../outputs/shap_summary.png)

`PageValues` güçlü bir sinyal olsa da gerçek zamanlı kullanımda leakage riski ayrıca değerlendirilmelidir. SHAP ilişkisel model katkısını gösterir, nedensellik göstermez.